# Load concept70_logits Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

OUT70 = Path('./data/concept_70')
train_path = OUT70 / 'concept70_logits_train.csv'
val_path   = OUT70 / 'concept70_logits_val.csv'
test_path  = OUT70 / 'concept70_logits_test.csv'
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
trainval_df = pd.concat([train_df, val_df], ignore_index=True)
feature_cols = [c for c in trainval_df.columns if c not in ['id','label','split']]
scaler = StandardScaler()
trainval_df[feature_cols] = scaler.fit_transform(trainval_df[feature_cols])
test_df[feature_cols]     = scaler.transform(test_df[feature_cols])
X_train = trainval_df[feature_cols].to_numpy(dtype=np.float32)
y_train = trainval_df['label'].to_numpy(dtype=np.int64)
X_test  = test_df[feature_cols].to_numpy(dtype=np.float32)
y_test  = test_df['label'].to_numpy(dtype=np.int64)

# Train Soft Decision Tree (SDT)

After training, save the weights to `./checkpoints/sdt_bloodmnist_concept70.pt`.

In [ ]:
# Soft Decision Tree (SDT) training and saving (based on concept70 features)
import os
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from SDT_pt import SDT
from torch.optim.lr_scheduler import StepLR

# Device
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
print('Using device:', device)

# Basic parameters and data loaders
input_dim = X_train.shape[1]
num_classes = int(np.max(y_train)) + 1

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds  = TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test))
trainloader = DataLoader(train_ds, batch_size=512, shuffle=True)
testloader  = DataLoader(test_ds,  batch_size=512)

# Build SDT model (weaker regularization to estimate model capacity first)
use_cuda = (device.type == 'cuda')
depth = 4
lamda = 5e-5          # disable balance penalty for now
inv_temp = 2       # keep temperature unchanged for now
tree = SDT(input_dim, num_classes, depth=depth, lamda=lamda, use_cuda=use_cuda, inv_temp=inv_temp,
           hard_leaf_inference=True,
           use_penalty_ema=False, penalty_ema_beta=0.9).to(device)

weight_decay=5e-4
# optimizer = torch.optim.Adam(tree.parameters(), lr=1e-3, weight_decay=0.0)
optimizer = torch.optim.Adam(tree.parameters(), lr=5e-3, weight_decay=weight_decay)
criterion = torch.nn.CrossEntropyLoss()
scheduler = StepLR(optimizer, step_size=100, gamma=0.7)
print(f"Device: {device}, CUDA available: {torch.cuda.is_available()} | use_cuda flag: {use_cuda}")

# Training / evaluation loop with simple early stopping
def train_and_eval(epochs=300, log_interval=100, early_stop_patience=50):
    best_acc = 0.0
    best_epoch = 0
    epochs_no_improve = 0

    for epoch in range(1, epochs+1):
        # Train
        tree.train()
        for batch_idx, (data, target) in enumerate(trainloader):
            data, target = data.to(device), target.to(device)
            logits, penalty = tree.forward(data, is_training_data=True)
            loss = criterion(logits, target) + penalty
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if batch_idx % log_interval == 0:
                with torch.no_grad():
                    pred = logits.argmax(dim=1)
                    correct = (pred == target).sum().item()
                print(f"Epoch {epoch:03d} | Batch {batch_idx:04d} | Loss {loss.item():.5f} | Correct {correct}/{data.size(0)}")

        # Learning rate decay
        scheduler.step()

        # Eval
        tree.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in testloader:
                data, target = data.to(device), target.to(device)
                logits = tree.forward(data)
                pred = logits.argmax(dim=1)
                correct += (pred == target).sum().item()
                total += target.size(0)
        acc = 100.0 * correct / total
        print(f"Epoch {epoch:03d} | Test Acc: {acc:.3f}% | Best: {best_acc:.3f}% (epoch {best_epoch:03d})")

        # Early stopping logic
        if acc > best_acc + 1e-4:  # significant improvement
            best_acc = acc
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if early_stop_patience is not None and epochs_no_improve >= early_stop_patience:
                print(f"Early stopping at epoch {epoch:03d} (no improvement for {early_stop_patience} epochs)")
                break

    return best_acc, best_epoch

Using device: cpu
Device: cpu, CUDA available: True | use_cuda flag: False


In [ ]:
epoch_num = 2000
best_acc, best_epoch = train_and_eval(epochs=epoch_num, early_stop_patience=200)
print('Best Acc: %.3f%% at epoch %d' % (best_acc, best_epoch))

Epoch 001 | Batch 0000 | Loss 2.08513 | Correct 89/512
Epoch 001 | Test Acc: 53.844% | Best: 0.000% (epoch 000)
Epoch 002 | Batch 0000 | Loss 1.87713 | Correct 294/512
Epoch 001 | Test Acc: 53.844% | Best: 0.000% (epoch 000)
Epoch 002 | Batch 0000 | Loss 1.87713 | Correct 294/512
Epoch 002 | Test Acc: 63.198% | Best: 53.844% (epoch 001)
Epoch 003 | Batch 0000 | Loss 1.70197 | Correct 327/512
Epoch 002 | Test Acc: 63.198% | Best: 53.844% (epoch 001)
Epoch 003 | Batch 0000 | Loss 1.70197 | Correct 327/512
Epoch 003 | Test Acc: 75.797% | Best: 63.198% (epoch 002)
Epoch 004 | Batch 0000 | Loss 1.50493 | Correct 398/512
Epoch 003 | Test Acc: 75.797% | Best: 63.198% (epoch 002)
Epoch 004 | Batch 0000 | Loss 1.50493 | Correct 398/512
Epoch 004 | Test Acc: 80.503% | Best: 75.797% (epoch 003)
Epoch 005 | Batch 0000 | Loss 1.30730 | Correct 424/512
Epoch 004 | Test Acc: 80.503% | Best: 75.797% (epoch 003)
Epoch 005 | Batch 0000 | Loss 1.30730 | Correct 424/512
Epoch 005 | Test Acc: 83.952% | Bes

In [ ]:
# Save checkpoint to the specified path
def save_checkpoint(path: str, tree, optimizer, extra=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    ckpt = {
        'model_state': tree.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'meta': {
            'input_dim': tree.input_dim,
            'output_dim': tree.output_dim,
            'depth': tree.depth,
            'lamda': tree.lamda,
            'inv_temp': tree.inv_temp,
            'hard_leaf_inference': tree.hard_leaf_inference,
        }
    }
    if extra is not None:
        ckpt['extra'] = extra
    torch.save(ckpt, path)
    print(f"Saved checkpoint to: {path}")

ckpt_path = "./checkpoints/sdt_bloodmnist_concept70.pt"
## Save after training is complete:
save_checkpoint(ckpt_path, tree, optimizer, extra={'note': f'best_acc={best_acc:.3f} at epoch {best_epoch}'})

Saved checkpoint to: ./checkpoints/sdt_bloodmnist_concept70.pt
